# 01 — QLoRA Training: Mistral-7B on ArXiv + SciTLDR

**Environment:** Google Colab (A100) or Kaggle (P100/T4)  
**Runtime:** ~2–3 hours for 3 epochs on 10k samples  

## Pipeline
1. Install Unsloth + dependencies
2. Load & process datasets (ArXiv + SciTLDR)
3. Load Mistral-7B-Instruct-v0.3 in 4-bit
4. Apply QLoRA adapters
5. Train with SFTTrainer
6. Save adapter weights
7. (Optional) Push to HuggingFace Hub

## 1. Environment Setup

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'No GPU detected!')

In [ ]:
# Install Unsloth (CUDA-specific)
# This uses the official Unsloth install command for Colab
import sys
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers rouge-score bert-score PyYAML
print('✅ Dependencies installed')

In [ ]:
# Mount Google Drive (optional — for saving checkpoints)
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/research-summarizer/checkpoints'

# Clone project (if running on fresh Colab)
import os
if not os.path.isdir('research-paper-summarizer'):
    !git clone https://github.com/YOUR_USERNAME/research-paper-summarizer.git
    %cd research-paper-summarizer
else:
    print('Project already cloned')

## 2. Load Config & Datasets

In [ ]:
import yaml
import sys
sys.path.insert(0, '.')

with open('config/training_config.yaml') as f:
    config = yaml.safe_load(f)

# Optionally override dataset sizes for faster experiments
# config['data']['arxiv_train_size'] = 2000

print('Config loaded:')
print(f'  Model: {config["model"]["name"]}')
print(f'  LoRA rank: {config["lora"]["r"]}')
print(f'  Learning rate: {config["training"]["learning_rate"]}')
print(f'  ArXiv train size: {config["data"]["arxiv_train_size"]}')

In [ ]:
from src.data.dataset_loader import load_datasets

train_raw, val_raw, test_raw = load_datasets(config)
print(f'Train: {len(train_raw):,}  |  Val: {len(val_raw):,}  |  Test: {len(test_raw):,}')
print('Sample:', train_raw[0]['source'][:200])

## 3. Load Mistral-7B in 4-bit (Unsloth)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config['model']['name'],
    max_seq_length=config['model']['max_seq_length'],
    dtype=None,          # auto-detect bfloat16 / float16
    load_in_4bit=True,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f'Model loaded: {config["model"]["name"]}')
print(f'Vocab size: {tokenizer.vocab_size:,}')

## 4. Apply QLoRA Adapters

In [ ]:
lora_cfg = config['lora']

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg['r'],
    lora_alpha=lora_cfg['lora_alpha'],
    lora_dropout=lora_cfg['lora_dropout'],
    target_modules=lora_cfg['target_modules'],
    bias=lora_cfg['bias'],
    use_gradient_checkpointing=lora_cfg['use_gradient_checkpointing'],
    random_state=config['data']['seed'],
    use_rslora=False,
)
model.print_trainable_parameters()

## 5. Prepare Training Dataset

In [ ]:
from src.data.preprocess import tokenize_dataset

train_ds = tokenize_dataset(train_raw, tokenizer, config, 'train')
val_ds   = tokenize_dataset(val_raw,   tokenizer, config, 'validation')

print('Sample formatted prompt:')
print(train_ds[0]['text'][:600])

## 6. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

tc = config['training']

training_args = TrainingArguments(
    output_dir=tc['output_dir'],
    num_train_epochs=tc['num_train_epochs'],
    per_device_train_batch_size=tc['per_device_train_batch_size'],
    per_device_eval_batch_size=tc['per_device_eval_batch_size'],
    gradient_accumulation_steps=tc['gradient_accumulation_steps'],
    warmup_steps=tc['warmup_steps'],
    learning_rate=tc['learning_rate'],
    weight_decay=tc['weight_decay'],
    lr_scheduler_type=tc['lr_scheduler_type'],
    optim=tc['optim'],
    bf16=True,
    logging_steps=tc['logging_steps'],
    eval_strategy='steps',
    eval_steps=tc['eval_steps'],
    save_steps=tc['save_steps'],
    save_total_limit=tc['save_total_limit'],
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    seed=config['data']['seed'],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=config['model']['max_seq_length'],
    packing=tc['packing'],
    args=training_args,
)

print('✅ Trainer ready. Starting training ...')

In [ ]:
trainer_stats = trainer.train()
print(f'Training complete!')
print(f'  Steps: {trainer_stats.global_step}')
print(f'  Training Loss: {trainer_stats.training_loss:.4f}')

## 7. Save Adapter Weights

In [ ]:
import os
OUTPUT_DIR = config['training']['output_dir']
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✅ Adapter saved to {OUTPUT_DIR}')
print(os.listdir(OUTPUT_DIR))

## 8. Quick Inference Test

In [ ]:
from unsloth import FastLanguageModel
from src.data.preprocess import INFERENCE_TEMPLATE
import torch

FastLanguageModel.for_inference(model)

test_paper = test_raw[0]['source'][:1500]
prompt = INFERENCE_TEMPLATE.format(source=test_paper)

inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False, repetition_penalty=1.1,
                         pad_token_id=tokenizer.eos_token_id)

input_len = inputs['input_ids'].shape[1]
summary = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
print('=== GENERATED SUMMARY ===')
print(summary)
print('\n=== REFERENCE ===')
print(test_raw[0]['target'])

## 9. (Optional) Push to HuggingFace Hub

In [ ]:
# Set HF_TOKEN in environment or use secrets
# import os
# from huggingface_hub import login
# login(token=os.environ['HF_TOKEN'])

# HF_REPO_ID = 'your-username/mistral7b-research-summarizer-qlora'
# model.push_to_hub(HF_REPO_ID)
# tokenizer.push_to_hub(HF_REPO_ID)
# print(f'Model pushed to https://huggingface.co/{HF_REPO_ID}')

print('Uncomment the lines above and set HF_TOKEN to push to Hub.')

## Next Steps
- ✅ Model trained and saved
- → Run `notebooks/02_evaluation.ipynb` to compute ROUGE-L, BERTScore, AlignScore